# ARIMA (AutoRegressive Integrated Moving Average)

**ARIMA** is a generalized statistical time series forecasting model designed to handle real-world sequential data that exhibits **upward or downward trends** (non-stationary data).

While the standard ARMA model has two parameters $(p, q)$, ARIMA introduces a third parameter **$d$**, denoted as **$\text{ARIMA}(p, d, q)$**:

* **$p$ (AutoRegressive):** Number of past (lagged) actual observations included in the model.
* **$d$ (Integrated):** Degree of **differencing** applied to eliminate trends and achieve stationarity.
* **$q$ (Moving Average):** Number of lagged forecast residual errors (shocks) included in the model.

## Differences Between ARMA and ARIMA

| Feature | ARMA$(p, q)$ | ARIMA$(p, d, q)$ |
| :--- | :--- | :--- |
| **Full Name** | AutoRegressive Moving Average | AutoRegressive **Integrated** Moving Average |
| **Parameters** | 2 parameters: $(p, q)$ | 3 parameters: $(p, d, q)$ |
| **Stationarity Requirement** | Data **must already be stationary** (constant mean and variance over time). | Handles **non-stationary data** by transforming it internally via differencing. |
| **Handles Trends?** | **No.** If data is trending up or down, ARMA produces invalid, unstable forecasts. | **Yes.** Differencing eliminates trends so the model can capture the true pattern. |
| **Anchor / Baseline** | Historical mean $\mu$ (reverts to past average). | Current actual level $Y_t$ (projects forward from the present). |
| **Long-Term Forecast Behavior** | Flatline at historical average (mean-reverting). | Can project continuous upward or downward growth trajectories. |
| **Primary Use Case** | Stationary anomalies, normalized returns, cyclical sensor data without trend. | Business KPIs, sales, revenues, stock prices, economic indicators with trends. |

### Key Conceptual Differences

1. **Stationarity Assumption:**
   ARMA requires the mean and variance to remain constant over the entire timeline. Real-world business data rarely adheres to this assumption. ARIMA eliminates this barrier by introducing the differencing parameter $d$.

2. **Prediction Target:**
   * **ARMA** directly models and predicts the **absolute value** of the variable: $\hat{Y}_t$.
   * **ARIMA** models and predicts the **incremental change** from period to period: $\widehat{\Delta^d Y}_t$, then adds that change back to the current level.

3. **Mean Reversion vs. Trend Continuation:**
   ARMA suffers from **false gravity** (mean reversion): whenever a series drifts away from its historical mean, ARMA pulls future predictions back down toward that mean. ARIMA allows the process to wander or trend without dragging it back to the past.

## What Does the "I" (Integrated / Differencing) Actually Do?

In mathematics, **integration is the reverse of differencing** (analogous to how integration is the inverse of differentiation in calculus).

When data has a trend, its average value is constantly rising or falling over time. Differencing stabilizes the mean by tracking the **relative change from one time step to the next** rather than the absolute level:

### 1. First-Order Differencing ($d = 1$)
Subtract yesterday's value from today's value:

$$\Delta Y_t = Y_t - Y_{t-1}$$

*Instead of predicting "How many total units will be recorded?", the model predicts "How much will the value increase or decrease compared to yesterday?"*

### 2. Second-Order Differencing ($d = 2$)
If the trend itself is accelerating (e.g., exponential growth), differencing is applied twice (differencing the differences):

$$\Delta^2 Y_t = \Delta Y_t - \Delta Y_{t-1} = (Y_t - Y_{t-1}) - (Y_{t-1} - Y_{t-2})$$

> In practice, **$d = 1$** is sufficient for over 90% of real-world trending datasets; $d > 2$ is rarely required and risks introducing artificial autocorrelation.

## How ARIMA Works (The 3-Stage Pipeline)

When you fit an $\text{ARIMA}(p, d, q)$ model, it automatically executes three stages behind the scenes:

```text
[ Non-Stationary Data (Trending) ]
               │
               ▼  Stage 1: Differencing (d)
[ Stationary Differenced Series (ΔY) ]
               │
               ▼  Stage 2: Fit ARMA(p, q)
[ Forecast of the Change (ΔŶ) ]
               │
               ▼  Stage 3: Integration (Invert Differencing)
[ Final Forecast on Original Scale (Ŷ) ]
```

1. **Stage 1 (Differencing - $d$):**
   It subtracts consecutive values $d$ times to strip away the trend, leaving a stationary series of changes $\Delta^d Y_t$.

2. **Stage 2 (ARMA Modeling - $p, q$):**
   It fits a standard $\text{ARMA}(p, q)$ model on those stationary changes using past changes (AR) and past residual errors/surprises (MA):
   $$\widehat{\Delta Y}_t = c + \sum_{i=1}^{p} \phi_i \Delta Y_{t-i} + \sum_{j=1}^{q} \theta_j e_{t-j}$$

3. **Stage 3 (Integration):**
   It inverts the differencing by accumulating the predicted changes back onto the last observed actual level:
   $$\hat{Y}_{\text{today}} = Y_{\text{yesterday}} + \widehat{\Delta Y}_{\text{today}}$$

## Mathematical Comparison for Time Step 2: ARMA(2, 2) vs. ARIMA(2, 1, 2)

Here we look back **two time steps** into history ($p = 2, q = 2$).

* **Observed Sales Data:**
  $$Y_1 = 100, \quad Y_2 = 110, \quad Y_3 = 120, \quad Y_4 = 130$$
* **Target:** Forecast Day 5 ($\hat{Y}_5$)
* **Model Parameters (Time Step 2):**
  * AR weights: $\phi_1 = 0.5$ (yesterday, $t-1$), $\phi_2 = 0.3$ (two days ago, $t-2$)
  * MA weights: $\theta_1 = 0.4$ (yesterday, $t-1$), $\theta_2 = 0.2$ (two days ago, $t-2$)
  * Past forecast errors (shocks): $\epsilon_4 = +15$ (yesterday), $\epsilon_3 = +10$ (two days ago)

### Part 1: Step-by-Step Math of ARMA(2, 2)

**General Equation:**
$$\hat{Y}_t = c + (\phi_1 Y_{t-1} + \phi_2 Y_{t-2}) + (\theta_1 \epsilon_{t-1} + \theta_2 \epsilon_{t-2})$$

1. **Compute Historical Mean ($\mu$):**
   $$\mu = \frac{100 + 110 + 120 + 130}{4} = \frac{460}{4} = 115$$

2. **Compute Baseline Constant ($c$):**
   $$c = \mu(1 - \phi_1 - \phi_2) = 115 \times (1 - 0.5 - 0.3) = 115 \times 0.2 = 23$$

   > **The term (1−weights) comes from a simple mathematical balance: the memory of past days and the baseline constant must add up to 100% of the long-term average. Which simply means sum of all the weights used (in baseline, AR, & in MA) should always be 1.**

3. **Compute Forecast for Day 5 ($\hat{Y}_5$):**
   $$\hat{Y}_5 = c + (\phi_1 Y_4 + \phi_2 Y_3) + (\theta_1 \epsilon_4 + \theta_2 \epsilon_3)$$
   Substitute values:
   * $c = 23$
   * $\phi_1 Y_4 = 0.5 \times 130 = 65$
   * $\phi_2 Y_3 = 0.3 \times 120 = 36$
   * $\theta_1 \epsilon_4 = 0.4 \times 15 = 6$
   * $\theta_2 \epsilon_3 = 0.2 \times 10 = 2$

   $$\hat{Y}_5 = 23 + (65 + 36) + (6 + 2)$$
   $$\hat{Y}_5 = 23 + 101 + 8$$
   $$\mathbf{\hat{Y}_5 = 132 \text{ cups}}$$

*(Notice: Despite sales growing by $+10$ every day and having positive shocks $+15$ and $+10$, ARMA predicts only $132$ cups because the constant $c = 23$ strongly pulls the forecast back toward the historical average of $115$).*

### Part 2: Step-by-Step Math of ARIMA(2, 1, 2)

Here, $d = 1$. The model operates on the series of first differences: $\Delta Y_t = Y_t - Y_{t-1}$.

1. **First-Order Differencing ($d = 1$):**
   $$\Delta Y_2 = Y_2 - Y_1 = 110 - 100 = 10$$
   $$\Delta Y_3 = Y_3 - Y_2 = 120 - 110 = 10$$
   $$\Delta Y_4 = Y_4 - Y_3 = 130 - 120 = 10$$
   $$\Delta Y = [10, 10, 10]$$

2. **Compute Mean of Differences ($\mu_{\Delta}$):**
   $$\mu_{\Delta} = \frac{10 + 10 + 10}{3} = 10$$

3. **Compute Differenced Baseline Constant ($c_{\Delta}$):**
   $$c_{\Delta} = \mu_{\Delta}(1 - \phi_1 - \phi_2) = 10 \times (1 - 0.5 - 0.3) = 10 \times 0.2 = 2$$

4. **Forecast the Difference ($\widehat{\Delta Y}_5$):**
   $$\widehat{\Delta Y}_t = c_{\Delta} + (\phi_1 \Delta Y_{t-1} + \phi_2 \Delta Y_{t-2}) + (\theta_1 \epsilon_{t-1} + \theta_2 \epsilon_{t-2})$$
   Substitute values:
   * $c_{\Delta} = 2$
   * $\phi_1 \Delta Y_4 = 0.5 \times 10 = 5$
   * $\phi_2 \Delta Y_3 = 0.3 \times 10 = 3$
   * $\theta_1 \epsilon_4 = 0.4 \times 15 = 6$
   * $\theta_2 \epsilon_3 = 0.2 \times 10 = 2$

   $$\widehat{\Delta Y}_5 = 2 + (5 + 3) + (6 + 2)$$
   $$\widehat{\Delta Y}_5 = 2 + 8 + 8 = \mathbf{18 \text{ cups}}$$

5. **The "I" (Integration) Step — Re-Accumulate to Original Scale:**
   $$\hat{Y}_5 = Y_4 + \widehat{\Delta Y}_5$$
   $$\hat{Y}_5 = 130 + 18$$
   $$\mathbf{\hat{Y}_5 = 148 \text{ cups}}$$

*(Notice: ARIMA anchors on $Y_4 = 130$, adds the expected $+10$ trend, plus $+8$ cups from the lingering shocks of the past two days, yielding a realistic $148$ cups).*

## The Exact Algebraic Difference the "I" Makes for Time Step 2

Comparing the expanded algebraic forms reveals why the "I" is so essential when looking back multiple time steps:

### In ARMA(2, 2):
$$\hat{Y}_5 = \mu + \phi_1(Y_4 - \mu) + \phi_2(Y_3 - \mu) + \theta_1 \epsilon_4 + \theta_2 \epsilon_3$$
$$\hat{Y}_5 = 115 + 0.5(130 - 115) + 0.3(120 - 115) + (6 + 2)$$
$$\hat{Y}_5 = 115 + 0.5(15) + 0.3(5) + 8$$
$$\hat{Y}_5 = 115 + 7.5 + 1.5 + 8 = \mathbf{132}$$

* **Anchor Point:** The historical mean $\mu = 115$.
* **Effect:** Both past actual deviations $(Y_4 - \mu = 15)$ and $(Y_3 - \mu = 5)$ are multiplied by fractions ($0.5$ and $0.3$). The model acts as a spring pulling predictions back down toward the past average $115$.

### In ARIMA(2, 1, 2):
$$\hat{Y}_5 = Y_4 + \mu_{\Delta} + \phi_1(\Delta Y_4 - \mu_{\Delta}) + \phi_2(\Delta Y_3 - \mu_{\Delta}) + \theta_1 \epsilon_4 + \theta_2 \epsilon_3$$
$$\hat{Y}_5 = 130 + 10 + 0.5(10 - 10) + 0.3(10 - 10) + (6 + 2)$$
$$\hat{Y}_5 = 130 + 10 + 0 + 0 + 8 = \mathbf{148}$$

* **Anchor Point:** Yesterday's actual sales level $Y_4 = 130$.
* **Effect:** The model projects forward from the latest operational reality. It adds the baseline drift rate ($+10$) and the multi-step shock ripples ($+8$) without dragging the business backward.

## Summary: When to Use ARMA vs. ARIMA

* **Use ARMA** when the data is strictly stationary (no upward/downward trend, constant mean and variance).
  * *Examples:* Daily temperature anomalies around seasonal averages, standardized returns, stationary sensor vibration residuals.

* **Use ARIMA** whenever the data displays a trend or evolving level over time.
  * *Examples:* Company revenue, website traffic, stock prices, retail sales, inflation indices, commodity demand.

## Advantages and Disadvantages of ARIMA (Pros & Cons)

### Advantages (Pros) of ARIMA

* **Handles Non-Stationary Trends Automatically:**
  By incorporating the differencing parameter ($d$), ARIMA eliminates the need for manual trend-filtering or detrending algorithms, enabling direct modeling of trending data.

* **Preserves Trend Trajectories:**
  Unlike ARMA (which forces predictions back to a flat historical mean), ARIMA anchors forecasts to the latest observed values, projecting ongoing growth or contraction accurately into the future.

* **High Accuracy for Short-to-Medium Horizons:**
  When applied to univariate series where autocorrelation and trend drive the dynamics, ARIMA consistently outperforms complex models and naive heuristics.

* **Model Parsimony (Prevents Overfitting):**
  A well-tuned $\text{ARIMA}(1, 1, 1)$ or $\text{ARIMA}(2, 1, 1)$ model uses only 3 to 5 parameters, drastically minimizing the risk of overfitting compared to high-capacity machine learning or deep neural models.

* **Dynamic Error Correction:**
  The moving average component ($q$) rapidly integrates recent forecast errors, adjusting subsequent predictions when unexpected external events occur.

* **Requires No External Features:**
  ARIMA generates forecasts purely from the internal historical patterns of the series, making it usable when exogenous covariates are unavailable or difficult to predict.

### Disadvantages (Cons) & Limitations of ARIMA

* **Lack of Built-In Seasonality:**
  Standard ARIMA cannot capture repeating periodic cycles (e.g., weekly, monthly, or quarterly seasonality). Datasets with seasonal spikes require upgrading to **SARIMA**.

* **Linearity Restriction:**
  ARIMA assumes that future values depend linearly on past values and errors. It fails on non-linear patterns, structural regime changes, or chaotic dynamical systems.

* **Danger of Over-Differencing:**
  Applying differencing when the data is already stationary ($d > 1$ when $d = 0$ or $1$ suffices) introduces artificial negative autocorrelation, inflates sample variance, and reduces forecasting accuracy.

* **Ambiguity in Hyperparameter Selection ($p, d, q$):**
  Selecting optimal $(p, d, q)$ orders via ACF/PACF visual inspection can be subjective. Automated grid searches (like `auto_arima`) can be time-consuming on very long series.

* **Rapidly Widening Prediction Intervals:**
  For long-term forecasting horizons, uncertainty compounds quickly. ARIMA forecasts tend to converge to a straight line with very wide confidence bounds.

* **Sensitive to Outliers and Missing Timestamps:**
  A single missing timestamp or sudden extreme outlier distorts the differencing operation, causing cascading errors across lagged terms.

## How to Use ARIMA in Code (Step-by-Step Python Guide)

The standard library for ARIMA in Python is `statsmodels` (via `statsmodels.tsa.arima.model.ARIMA`), often paired with `pmdarima` for automated parameter selection.

### Step 1: Import Required Libraries

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
import pmdarima as pm

%matplotlib inline
```

### Step 2: Load and Prepare Time Series Data

Ensure the index is a monotonic `DatetimeIndex` with an explicit frequency:

```python
# Load time series data
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, index_col='Month', parse_dates=True)

# Set explicit frequency (MS = Month Start)
df.index.freq = 'MS'
series = df['Passengers']

# Visualize raw series
plt.figure(figsize=(10, 4))
plt.plot(series, label='Observed Data')
plt.title('Original Time Series (Non-Stationary Trend)')
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend()
plt.show()
```

### Step 3: Determine Differencing Order ($d$) Using ADF Test

```python
def check_stationarity(timeseries):
    result = adfuller(timeseries.dropna())
    print(f'ADF Statistic: {result[0]:.4f}')
    print(f'p-value: {result[1]:.4f}')
    if result[1] <= 0.05:
        print('Result: The series is STATIONARY (p <= 0.05)')
    else:
        print('Result: The series is NON-STATIONARY (p > 0.05) -> Differencing needed')

# Test raw series (d = 0)
print('--- Testing Raw Series ---')
check_stationarity(series)

# Apply first-order differencing (d = 1)
diff_1 = series.diff().dropna()
print('\n--- Testing 1st Differenced Series ---')
check_stationarity(diff_1)
```

### Step 4: Identify Orders $p$ and $q$

#### Method A: Visual Inspection via ACF & PACF Plots
```python
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# PACF plot helps identify p (AR order)
plot_pacf(diff_1, lags=20, ax=axes[0], title='Partial Autocorrelation (PACF) -> Determines p')

# ACF plot helps identify q (MA order)
plot_acf(diff_1, lags=20, ax=axes[1], title='Autocorrelation (ACF) -> Determines q')

plt.tight_layout()
plt.show()
```

#### Method B: Automated Search via Auto-ARIMA
```python
# Automatically search for the best (p, d, q) minimizing AIC/BIC
auto_model = pm.auto_arima(
    series,
    start_p=0, max_p=3,
    start_q=0, max_q=3,
    d=1,
    seasonal=False,
    trace=True,
    error_action='ignore',
    suppress_warnings=True,
    stepwise=True
)
print('Best Model Order:', auto_model.order)
```

### Step 5: Fit the ARIMA Model in Statsmodels

Pass the selected order $(p, d, q)$ directly to `ARIMA`:

```python
# Train/Test Split (e.g., hold out last 12 months for testing)
train = series.iloc[:-12]
test = series.iloc[-12:]

# Fit ARIMA(p, d, q) model on training data
model = ARIMA(train, order=(2, 1, 2))
fitted_model = model.fit()

# Print full statistical summary
print(fitted_model.summary())
```

### Step 6: Model Diagnostics & Residual Checking

Good time series models should have **residuals that behave like white noise** (uncorrelated with zero mean):

```python
# Plot residual diagnostics (standardized residuals, histogram, Q-Q plot, correlogram)
fitted_model.plot_diagnostics(figsize=(12, 8))
plt.tight_layout()
plt.show()
```

### Step 7: Out-of-Sample Forecasting & Visualizing Prediction Intervals

```python
# Generate out-of-sample forecasts with 95% confidence intervals
forecast_obj = fitted_model.get_forecast(steps=len(test))
forecast_values = forecast_obj.predicted_mean
conf_int = forecast_obj.conf_int(alpha=0.05)

# Plot historical train, actual test, and forecast
plt.figure(figsize=(12, 5))
plt.plot(train.index, train, label='Train (History)')
plt.plot(test.index, test, label='Test (Actual)', color='green')
plt.plot(test.index, forecast_values, label='ARIMA Forecast', color='red', linestyle='--')

# Fill 95% prediction interval
plt.fill_between(
    test.index,
    conf_int.iloc[:, 0],
    conf_int.iloc[:, 1],
    color='pink',
    alpha=0.3,
    label='95% Confidence Interval'
)

plt.title('ARIMA Out-of-Sample Forecast vs. Actuals')
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend(loc='upper left')
plt.show()
```

### Step 8: Evaluate Forecast Accuracy

```python
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

mae = mean_absolute_error(test, forecast_values)
rmse = np.sqrt(mean_squared_error(test, forecast_values))
mape = mean_absolute_percentage_error(test, forecast_values)

print(f'MAE:  {mae:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'MAPE: {mape:.2%}')
```